# Primary Figure — Concordance Ladder

**Single-panel main manuscript figure.** Answers one question: when two independent
projection methods (Ensembl, CAT) are applied to the same 400+ human genome assemblies,
do they agree, and at what resolution does agreement break down?

### Design principles
1. **Strict subset funnel** — each rung conditions on the one above. The denominator
   shrinks monotonically so every percentage is directly interpretable as
   "of the features that passed the previous filter, what fraction pass this one?"
2. **Per-assembly distributions** — every rung shows the spread across all assemblies
   (dot strip with median + IQR), not just the pangenome aggregate. This makes
   both claims simultaneously: concordance is high *and* consistent.
3. **No method comparison** — this figure is about agreement, not which method is
   better. Method-specific breakdowns belong in the supplementary.

### Ladder rungs
| Rung | Question | Denominator |
|------|----------|-------------|
| 1. Gene presence | Of all loci detected by either method, what fraction found by both? | Union of gene names per assembly |
| 2. Reciprocal locus overlap | Of genes present in both, what fraction have ≥95% reciprocal body overlap? | Rung 1 passers (= RBH pairs) |
| 3. Exact transcript match | Of overlapping gene pairs, what fraction share ≥1 exactly matching transcript? | Rung 2 passers |
| 4. CDS boundary concordance | Of those with an exact transcript, what fraction have identical start+stop codons? | Rung 3 passers (protein-coding subset) |
| 5. Frame integrity | Of CDS-concordant genes, what fraction have no detected frameshift? | Rung 4 passers |

### Data requirements
This notebook reads **pre-computed summary TSVs** produced by the main QC report
notebook (`hprc_annotation_qc_report.ipynb`), plus raw per-assembly pipeline outputs
where per-assembly breakdowns are needed. It does *not* reprocess GFF files.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from pathlib import Path
import warnings
import gc

warnings.filterwarnings('ignore')

# Nature-style defaults
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 8,
    'axes.labelsize': 9,
    'axes.titlesize': 10,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 7,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'pdf.fonttype': 42,      # editable text in Illustrator
    'ps.fonttype': 42,
})

print(f'pandas {pd.__version__}, numpy {np.__version__}')

## Configuration

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
# Adjust these to point at the pipeline output directory.
OUTPUT_DIR  = Path('../results')
QC_DIR      = OUTPUT_DIR / 'qc_metrics'
RESULTS_DIR = OUTPUT_DIR / 'results'
SUMMARY_DIR = OUTPUT_DIR / 'summary_stats'
FIGURE_DIR  = OUTPUT_DIR / 'figures'
FIGURE_DIR.mkdir(exist_ok=True, parents=True)

CHUNK_SIZE = 50  # for chunked file reading

# ── Colour palette ─────────────────────────────────────────────────────────
GENE_COL = '#2a9d8f'
TX_COL   = '#457b9d'
CDS_COL  = '#e76f51'

print(f'Output:  {OUTPUT_DIR}')
print(f'Figures: {FIGURE_DIR}')

---
## 1. Compute per-assembly funnel metrics

Each rung is computed **per assembly** so we can show distributions.
The funnel logic means each rung's denominator is the previous rung's numerator.

### 1a. Rung 1 — Gene presence (per assembly)

The existing `gene_presence_summary.tsv` only stores global aggregates.
We need per-assembly fractions, so we re-read the raw files.

In [ ]:
def compute_gene_presence_per_assembly():
    """
    For each assembly, compute:
      n_union  = number of unique gene names detected by either method
      n_both   = number detected by both
      pct_both = n_both / n_union
    """
    files = sorted(QC_DIR.rglob('*_gene_presence.tsv'))
    print(f'Found {len(files)} gene_presence files')
    if not files:
        return pd.DataFrame()

    rows = []
    for i in range(0, len(files), CHUNK_SIZE):
        chunk_files = files[i:i + CHUNK_SIZE]
        chunk_dfs = []
        for f in chunk_files:
            try:
                df = pd.read_csv(f, sep='\t')
                for col in ['present_in_ensembl', 'present_in_cat']:
                    df[col] = df[col].map(
                        {'True': True, 'False': False, True: True, False: False}
                    )
                # Exclude bare ENSG stable IDs (no display name)
                df = df[~df['gene_name'].str.match(r'^ENSG', na=False)]
                chunk_dfs.append(df)
            except Exception as e:
                print(f'  error: {f.name}: {e}')

        if chunk_dfs:
            chunk = pd.concat(chunk_dfs, ignore_index=True)
            for acc, grp in chunk.groupby('assembly_accession'):
                n_union = len(grp)
                n_both = ((grp['present_in_ensembl']) & (grp['present_in_cat'])).sum()
                rows.append({
                    'assembly_accession': acc,
                    'n_union_loci': n_union,
                    'n_both_loci': int(n_both),
                    'pct_gene_presence': n_both / n_union if n_union > 0 else np.nan,
                })
            del chunk, chunk_dfs
            gc.collect()

    out = pd.DataFrame(rows)
    out.to_csv(SUMMARY_DIR / 'funnel_rung1_gene_presence_per_asm.tsv',
               sep='\t', index=False)
    print(f'  => {len(out)} assemblies')
    return out

rung1 = compute_gene_presence_per_assembly()

### 1b. Rung 2 — Reciprocal locus overlap ≥95% (per assembly)

Of genes found by both (rung 1 passers), what fraction form an RBH pair
with ≥95% reciprocal gene-body overlap (Identical or Near-identical
in `classification_detailed`)?

Denominator = number of genes present in both annotations per assembly.
Numerator = RBH pairs classified as Identical or Near-identical.

In [ ]:
def compute_reciprocal_overlap_per_assembly():
    """
    Per assembly: of genes present in both methods, what fraction are
    Identical or Near-identical (>=95% reciprocal overlap) RBH pairs?
    """
    files = sorted(RESULTS_DIR.rglob('*.gene_pairs_rbh.tsv'))
    print(f'Found {len(files)} RBH files')
    if not files:
        return pd.DataFrame()

    HIGH_CLASSES = {'Identical', 'Near-identical'}
    rows = []

    for i in range(0, len(files), CHUNK_SIZE):
        chunk_files = files[i:i + CHUNK_SIZE]
        chunk_dfs = []
        for f in chunk_files:
            try:
                chunk_dfs.append(pd.read_csv(f, sep='\t'))
            except Exception as e:
                print(f'  error: {f.name}: {e}')

        if chunk_dfs:
            chunk = pd.concat(chunk_dfs, ignore_index=True)
            cls_col = ('classification_detailed'
                       if 'classification_detailed' in chunk.columns
                       else 'classification')

            for acc, grp in chunk.groupby('assembly_accession'):
                n_rbh = len(grp)
                n_high = grp[cls_col].isin(HIGH_CLASSES).sum()
                rows.append({
                    'assembly_accession': acc,
                    'n_rbh_pairs': n_rbh,
                    'n_high_overlap': int(n_high),
                    'pct_high_overlap': n_high / n_rbh if n_rbh > 0 else np.nan,
                })
            del chunk, chunk_dfs
            gc.collect()

    out = pd.DataFrame(rows)
    out.to_csv(SUMMARY_DIR / 'funnel_rung2_reciprocal_overlap_per_asm.tsv',
               sep='\t', index=False)
    print(f'  => {len(out)} assemblies')
    return out

rung2 = compute_reciprocal_overlap_per_assembly()

### 1c. Rung 3 — Exact transcript structure match (per assembly)

Of gene pairs passing rung 2 (≥95% overlap), what fraction share at
least one transcript with an exact exon-structure match?

Binary per-gene: `n_ens_exact >= 1`.

**Note:** The transcript concordance files are keyed to RBH pairs, not
filtered to the ≥95% subset. For the true funnel we need to intersect
with rung 2 passers. If the ≥95% RBH ID set is not available per row
in the transcript file, we use all RBH pairs as the denominator and
document this as a slight relaxation of the strict funnel.

In [ ]:
def compute_exact_transcript_per_assembly():
    """
    Per assembly: of RBH gene pairs, what fraction have at least one
    exact transcript structure match (n_ens_exact >= 1)?

    This is a binary per-gene metric, not the mean concordance rate.
    """
    files = sorted(QC_DIR.rglob('*_transcript_concordance.tsv'))
    print(f'Found {len(files)} transcript concordance files')
    if not files:
        return pd.DataFrame()

    rows = []
    for i in range(0, len(files), CHUNK_SIZE):
        chunk_files = files[i:i + CHUNK_SIZE]
        chunk_dfs = []
        for f in chunk_files:
            try:
                chunk_dfs.append(pd.read_csv(f, sep='\t'))
            except Exception as e:
                print(f'  error: {f.name}: {e}')

        if chunk_dfs:
            chunk = pd.concat(chunk_dfs, ignore_index=True)

            for acc, grp in chunk.groupby('assembly_accession'):
                n_genes = len(grp)
                # Binary: does this gene have at least one exact match?
                has_exact = (grp['n_ens_exact'] >= 1).sum()
                rows.append({
                    'assembly_accession': acc,
                    'n_genes_with_tx_data': n_genes,
                    'n_has_exact_tx': int(has_exact),
                    'pct_exact_transcript': has_exact / n_genes if n_genes > 0 else np.nan,
                })
            del chunk, chunk_dfs
            gc.collect()

    out = pd.DataFrame(rows)
    out.to_csv(SUMMARY_DIR / 'funnel_rung3_exact_transcript_per_asm.tsv',
               sep='\t', index=False)
    print(f'  => {len(out)} assemblies')
    return out

rung3 = compute_exact_transcript_per_assembly()

### 1d. Rungs 4 & 5 — CDS boundaries and frame integrity (per assembly)

These are already computed per-assembly in the main notebook
(`coding_integrity_per_assembly.tsv`). We reload and reformat.

In [ ]:
def load_coding_integrity_per_assembly():
    """
    Load or recompute per-assembly CDS concordance and frame integrity.

    Rung 4: start_stop_agreement (both codons match)
    Rung 5: 1 - (n_frameshifts / n_protein_coding)
    """
    path = SUMMARY_DIR / 'coding_integrity_per_assembly.tsv'
    if path.exists():
        df = pd.read_csv(path, sep='\t')
        print(f'Loaded coding integrity per-assembly: {len(df)} assemblies')
    else:
        # Recompute from raw files
        print('coding_integrity_per_assembly.tsv not found; recomputing...')
        files = sorted(QC_DIR.rglob('*_coding_integrity.tsv'))
        print(f'  Found {len(files)} coding integrity files')
        if not files:
            return pd.DataFrame()

        per_asm = []
        for i in range(0, len(files), CHUNK_SIZE):
            chunk_files = files[i:i + CHUNK_SIZE]
            chunk_dfs = []
            for f in chunk_files:
                try:
                    d = pd.read_csv(f, sep='\t')
                    for col in ['start_codon_match', 'stop_codon_match',
                                'frameshift_detected', 'has_ensembl_cds', 'has_cat_cds']:
                        if col in d.columns:
                            d[col] = d[col].map(
                                {'True': True, 'False': False, True: True, False: False}
                            )
                    chunk_dfs.append(d)
                except Exception as e:
                    print(f'  error: {f.name}: {e}')

            if chunk_dfs:
                ch = pd.concat(chunk_dfs, ignore_index=True)
                with_cds = ch[
                    (ch['has_ensembl_cds'] == True) &
                    (ch['has_cat_cds'] == True)
                ]
                if len(with_cds) > 0:
                    stats = with_cds.groupby('assembly_accession').apply(
                        lambda x: pd.Series({
                            'n_protein_coding': len(x),
                            'start_stop_agreement': (
                                (x['start_codon_match'] == True) &
                                (x['stop_codon_match'] == True)
                            ).sum() / len(x),
                            'n_frameshifts': (x['frameshift_detected'] == True).sum(),
                        })
                    ).reset_index()
                    per_asm.append(stats)
                del ch, chunk_dfs, with_cds
                gc.collect()

        df = pd.concat(per_asm, ignore_index=True) if per_asm else pd.DataFrame()
        df.to_csv(path, sep='\t', index=False)
        print(f'  => {len(df)} assemblies')

    # Derive rung metrics
    if not df.empty:
        df['pct_cds_concordance'] = df['start_stop_agreement']
        df['pct_frame_integrity'] = 1.0 - (
            df['n_frameshifts'] / df['n_protein_coding']
        ).clip(0, 1)
    return df

coding_per_asm = load_coding_integrity_per_assembly()
coding_per_asm.head()

### 1e. Merge all rungs into a single per-assembly table

In [ ]:
# Merge on assembly_accession
funnel = rung1[['assembly_accession', 'pct_gene_presence']].copy()

if not rung2.empty:
    funnel = funnel.merge(
        rung2[['assembly_accession', 'pct_high_overlap']],
        on='assembly_accession', how='left'
    )
else:
    funnel['pct_high_overlap'] = np.nan

if not rung3.empty:
    funnel = funnel.merge(
        rung3[['assembly_accession', 'pct_exact_transcript']],
        on='assembly_accession', how='left'
    )
else:
    funnel['pct_exact_transcript'] = np.nan

if not coding_per_asm.empty:
    funnel = funnel.merge(
        coding_per_asm[['assembly_accession', 'pct_cds_concordance',
                        'pct_frame_integrity']],
        on='assembly_accession', how='left'
    )
else:
    funnel['pct_cds_concordance'] = np.nan
    funnel['pct_frame_integrity'] = np.nan

# Save
funnel.to_csv(SUMMARY_DIR / 'funnel_per_assembly_all_rungs.tsv',
              sep='\t', index=False)

print(f'Funnel table: {len(funnel)} assemblies x {len(funnel.columns)} columns')
print()
print(funnel.describe().T[['mean', '50%', 'min', 'max']].rename(
    columns={'50%': 'median'}
).round(4))

---
## 2. Compute pangenome aggregate (for bar values)

The bar length is the **median** across assemblies (robust to outliers).
The dot strip shows the full per-assembly distribution.

In [ ]:
# Ladder definition: (label, column_name, colour, level_label)
LADDER = [
    ('Gene loci detected by both methods',          'pct_gene_presence',    GENE_COL, 'Gene'),
    ('Reciprocal body overlap \u226595%',           'pct_high_overlap',     GENE_COL, 'Gene'),
    ('Share \u22651 exact transcript structure',    'pct_exact_transcript', TX_COL,   'Transcript'),
    ('Start and stop codons match',                 'pct_cds_concordance',  CDS_COL,  'CDS'),
    ('No frameshift detected',                      'pct_frame_integrity',  CDS_COL,  'CDS'),
]

# Pre-compute summary stats for annotation
ladder_stats = []
for label, col, colour, level in LADDER:
    vals = funnel[col].dropna() * 100  # convert to percent
    ladder_stats.append({
        'label': label,
        'col': col,
        'colour': colour,
        'level': level,
        'median': vals.median(),
        'mean': vals.mean(),
        'q25': vals.quantile(0.25),
        'q75': vals.quantile(0.75),
        'min': vals.min(),
        'max': vals.max(),
        'n_assemblies': len(vals),
    })

ladder_df = pd.DataFrame(ladder_stats)
print(ladder_df[['label', 'median', 'q25', 'q75', 'n_assemblies']].to_string(index=False))

---
## 3. Draw the figure

Layout: horizontal ladder bars (left panel, \u2154 width) + aligned dot-strip
distributions (right panel, \u2153 width). One row per rung.

In [ ]:
n_rungs = len(LADDER)
fig_h   = max(3.5, 0.75 * n_rungs + 1.0)  # scale height with rungs

fig = plt.figure(figsize=(7.2, fig_h))  # Nature single-column = 89 mm \u2248 3.5 in
                                          # double-column = 183 mm \u2248 7.2 in
gs  = GridSpec(1, 2, figure=fig, width_ratios=[2.2, 1], wspace=0.05)
ax_bar  = fig.add_subplot(gs[0])
ax_dot  = fig.add_subplot(gs[1], sharey=ax_bar)

y_pos = np.arange(n_rungs)

# ── Left panel: median bars ────────────────────────────────────────────────
for i, row in ladder_df.iterrows():
    ax_bar.barh(
        i, row['median'],
        color=row['colour'], edgecolor='white', linewidth=0.6,
        height=0.55, alpha=0.85, zorder=2,
    )
    # Median value label
    ax_bar.text(
        row['median'] + 0.4, i, f"{row['median']:.1f}%",
        va='center', ha='left', fontsize=7.5, fontweight='bold',
        color=row['colour'], zorder=3,
    )

# Hierarchy bracket lines
# Group consecutive rungs by level
levels_seen = []
for i, row in ladder_df.iterrows():
    if not levels_seen or levels_seen[-1][0] != row['level']:
        levels_seen.append((row['level'], row['colour'], i, i))
    else:
        levels_seen[-1] = (row['level'], row['colour'],
                           levels_seen[-1][2], i)

for level_label, col, y_lo, y_hi in levels_seen:
    ax_bar.plot(
        [-2.5, -2.5], [y_lo - 0.32, y_hi + 0.32],
        color=col, lw=3, solid_capstyle='round', clip_on=False,
    )
    ax_bar.text(
        -3.2, (y_lo + y_hi) / 2, level_label,
        color=col, fontsize=7.5, fontweight='bold',
        ha='right', va='center', clip_on=False,
    )

ax_bar.set_xlim(0, 108)
ax_bar.set_ylim(-0.6, n_rungs - 0.4)
ax_bar.set_yticks(y_pos)
ax_bar.set_yticklabels(ladder_df['label'], fontsize=8)
ax_bar.set_xlabel('Agreement between Ensembl and CAT (%)', fontsize=9)
ax_bar.xaxis.grid(True, alpha=0.3, linestyle='--', zorder=0)
ax_bar.set_axisbelow(True)
ax_bar.spines[['top', 'right', 'left']].set_visible(False)
ax_bar.tick_params(left=False)

# Legend
legend_patches = []
for level_label, col, _, _ in levels_seen:
    legend_patches.append(
        mpatches.Patch(color=col, label=f'{level_label} level')
    )
ax_bar.legend(
    handles=legend_patches, loc='lower right',
    fontsize=7, framealpha=0.9, edgecolor='none',
)

# ── Right panel: dot strips ────────────────────────────────────────────────
for i, (_, row_meta) in enumerate(ladder_df.iterrows()):
    col_name = row_meta['col']
    vals = funnel[col_name].dropna() * 100
    colour = row_meta['colour']

    # Jitter y for visibility
    jitter = np.random.default_rng(42).uniform(-0.18, 0.18, size=len(vals))

    ax_dot.scatter(
        vals, i + jitter,
        s=3, alpha=0.35, color=colour, edgecolors='none',
        rasterized=True, zorder=2,
    )

    # Median + IQR
    med = vals.median()
    q25, q75 = vals.quantile(0.25), vals.quantile(0.75)
    ax_dot.plot([q25, q75], [i, i], color='black', lw=1.5, zorder=3)
    ax_dot.plot(med, i, 'o', color='white', markersize=4,
                markeredgecolor='black', markeredgewidth=1, zorder=4)

ax_dot.set_xlabel('Per-assembly (%)\nMedian \u25cb  IQR \u2014', fontsize=7.5)
ax_dot.set_xlim(ax_bar.get_xlim())  # match x range
ax_dot.xaxis.grid(True, alpha=0.3, linestyle='--', zorder=0)
ax_dot.set_axisbelow(True)
ax_dot.spines[['top', 'right', 'left']].set_visible(False)
ax_dot.tick_params(left=False, labelleft=False)

# Annotate n assemblies
ax_dot.text(
    0.97, 0.97,
    f'n = {int(ladder_df["n_assemblies"].max()):,}\nassemblies',
    transform=ax_dot.transAxes, fontsize=6.5,
    ha='right', va='top', color='#555',
)

# ── Title ───────────────────────────────────────────────────────────────────
fig.suptitle(
    'Independent projection methods show high concordance\n'
    'across the human pangenome annotation hierarchy',
    fontsize=10.5, fontweight='bold', y=1.02,
)

# ── Save ────────────────────────────────────────────────────────────────────
for ext in ['png', 'pdf']:
    fig.savefig(FIGURE_DIR / f'figure_concordance_ladder.{ext}',
                dpi=300, bbox_inches='tight')

plt.show()
print('\u2713 Saved figure_concordance_ladder.{png,pdf}')

---
## 4. Outlier identification

Flag assemblies that are outliers (>1.5 IQR below Q1) on any rung.
These may correspond to assembly quality issues, genuinely divergent
haplotypes, or edge cases in one method's projection logic.

In [ ]:
outlier_flags = pd.DataFrame({'assembly_accession': funnel['assembly_accession']})

for _, row_meta in ladder_df.iterrows():
    col = row_meta['col']
    vals = funnel[col].dropna() * 100
    q25, q75 = vals.quantile(0.25), vals.quantile(0.75)
    iqr = q75 - q25
    lower_fence = q25 - 1.5 * iqr

    outlier_flags[f'outlier_{col}'] = (funnel[col] * 100) < lower_fence

# Any assembly flagged on any rung
outlier_cols = [c for c in outlier_flags.columns if c.startswith('outlier_')]
outlier_flags['any_outlier'] = outlier_flags[outlier_cols].any(axis=1)

n_outliers = outlier_flags['any_outlier'].sum()
print(f'Assemblies flagged as outlier on \u22651 rung: {n_outliers} / {len(outlier_flags)}')

if n_outliers > 0 and n_outliers <= 20:
    flagged = funnel.loc[
        outlier_flags['any_outlier'],
        ['assembly_accession'] + [r['col'] for r in ladder_stats]
    ].copy()
    for col in [r['col'] for r in ladder_stats]:
        flagged[col] = (flagged[col] * 100).round(1)
    print(flagged.to_string(index=False))
elif n_outliers > 20:
    print(f'  (too many to list — check funnel_per_assembly_all_rungs.tsv)')

outlier_flags.to_csv(
    SUMMARY_DIR / 'funnel_outlier_flags.tsv', sep='\t', index=False
)

---
## 5. Caption text block

Auto-generated draft caption with key numbers filled in.

In [ ]:
n_asm = int(ladder_df['n_assemblies'].max())
r1_med = ladder_df.loc[0, 'median']
r5_med = ladder_df.loc[len(ladder_df) - 1, 'median']
r1_iqr = ladder_df.loc[0, 'q75'] - ladder_df.loc[0, 'q25']

caption = f"""\
**Figure X. Ensembl and CAT annotation projections converge across the human
pangenome.**  Concordance between Ensembl (linear projection) and CAT
(graph-based projection) annotations is assessed at five hierarchical levels
across {n_asm:,} HPRC assemblies.  Left: each bar shows the median percentage
of features passing that concordance test (strict subset funnel — each rung
conditions on the one above).  Right: per-assembly distributions (one dot per
assembly; open circle = median; bar = interquartile range).  Gene-level
agreement (top rung) is {r1_med:.1f}% (IQR {r1_iqr:.1f} pp); even at the
most stringent test — frame integrity of CDS-concordant protein-coding genes
— median agreement remains {r5_med:.1f}%.  Gene identity is defined by shared
gene name (HGNC symbol); locus overlap uses \u226595% reciprocal gene-body
coverage from reciprocal best-hit (RBH) pairing; frameshift is inferred from
CDS length difference not divisible by 3 (3 bp tolerance on codon boundaries).
See Supplementary Figure X for per-rung breakdowns and method-specific details.
"""

print(caption)

---
## Notes for revision

- **Threshold sensitivity (rung 2):** consider a supplementary showing concordance
  at 90/95/99% overlap thresholds. If robust, cite in legend; if not, discuss.
- **Frame integrity vs assembly quality:** if assembly QV scores are available,
  correlating rung 5 with QV preempts a reviewer question.
- **Denominator definitions:** rung 1 uses gene-name matching (HGNC symbols);
  document in methods. Multi-mapped / one-to-many projections are handled in
  the supplementary figure.
- **Strict funnel relaxation:** rung 3 currently uses all RBH pairs as denominator
  rather than strict rung-2 passers, because the transcript concordance pipeline
  output is keyed to RBH pairs without the ≥95% filter. To make this exact,
  filter the transcript concordance files to only gene pairs that are
  Identical/Near-identical in the RBH overlap classification.